In [ ]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
from google.colab import userdata            # Colab's secret store

openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key   # IMPORTANT: the Agents SDK reads the key from here

openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:5])


OpenAI client successfully configured.
sk-pr


In [ ]:
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 293.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 271.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 312.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 308.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 252.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: packaging
^C


In [ ]:
from agents import Agent

fact_checker_instructions = """
Context:
You are a fact-checker who verifies the accuracy of statements.

Instructions:
When given a statement, carefully analyze its factual accuracy using your knowledge.

Input:
You will receive a statement that requires fact-checking.

Output:
Respond with:
1. A verdict prefix: either "✅ TRUE:" or "❌ FALSE:"
2. A brief, one-sentence explanation justifying your conclusion
3. if the input is not a fact checking question and its a general question donot use any verdict prefix, give just a plane response
"""


In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",                    # badge name
                           instructions = fact_checker_instructions, # job description
                           model = "gpt-4.1-mini")                   # which brain

print(f"Agent '{fact_checker_agent.name}' created successfully!")


Agent 'Fact Checker' created successfully!


In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))


In [ ]:
from agents import Runner

statement = "The Great Wall of China is visible from space with the naked eye."
print_markdown(f"Asking the Fact Checker to verify: '{statement}'")

response = await Runner.run(
    starting_agent = fact_checker_agent,   # WHICH agent to run
    input = statement                      # WHAT to give it
)

print_markdown("\n🤖 Agent's Response:\n")
print_markdown(response.final_output)      # the agent's final answer


Asking the Fact Checker to verify: 'The Great Wall of China is visible from space with the naked eye.'


🤖 Agent's Response:


❌ FALSE: The Great Wall of China is generally not visible from space with the naked eye due to its narrow width and natural coloration blending with the surroundings.

In [ ]:
check_recall_statement = "What did we discuss in the last message?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)


You asked me to fact-check statements by verifying their accuracy and provide a verdict with a brief explanation.

In [ ]:
check_recall_statement = "Do you remmember the fact check statement/question?"
response = await Runner.run(starting_agent = fact_checker_agent, input = check_recall_statement)
print_markdown(response.final_output)


I remember that I am supposed to fact-check statements by verifying their accuracy and providing a verdict prefix along with a brief explanation.

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = "gpt-6.1-sol")

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)


✅ TRUE: Mount Everest is the world’s tallest mountain above sea level, reaching approximately 8,849 metres (29,032 feet).

In [ ]:
fact_checker_agent = Agent(name = "Fact Checker",
                           instructions = fact_checker_instructions,
                           model = "gpt-5-mini")

statement = "The tallest mountain in the world is Mount Everst"
response = await Runner.run(starting_agent = fact_checker_agent, input = statement)
print_markdown(response.final_output)


✅ TRUE: Mount Everest (correct spelling) is the highest mountain above sea level at 8,848.86 m, although Mauna Kea is taller if measured from its base on the ocean floor.

In [ ]:
tweet_bot_instructions = """
Instruction:
Write a short, engaging tweet (≤280 characters) about the given topic. The tweet must
include at least one relevant emoji and one relevant hashtag.

Context:
You are **TweetBot**, a witty and creative social-media expert known for crafting viral tweets.

Input:
A topic provided by the user.

Output:
A single tweet (≤280 characters) about the topic, containing at least one relevant emoji
and one relevant hashtag.
"""

tweet_bot_agent = Agent(name = "Tweet Bot",
                        instructions = tweet_bot_instructions,
                        model = "gpt-5-mini")

topic = "The future of renewable energy"
tweet_response = await Runner.run(starting_agent = tweet_bot_agent, input = topic)
print_markdown(tweet_response.final_output)


The future of renewable energy is bright—literally. Solar, wind, batteries & smart grids teaming up to keep the lights on and the planet cool ⚡️🌞🌬️🔋🌍 #CleanEnergy